# Heatwave Model

### 1. Loading  libraries cleaned data 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("cleaned_climate_data.csv")

### 2. Define heatwave threshold

In [2]:
# Use summer months to define the 95 percentile threshold
summer_mask = df["MONTH"].isin([6, 7, 8])
summer_tmax = df.loc[summer_mask, "TMAX"].dropna()

if len(summer_tmax) == 0:
    raise ValueError("No summer TMAX values found. Check your data.")

heatwave_threshold = summer_tmax.quantile(0.95)

print(f"Heatwave threshold (95th percentile of summer TMAX): {heatwave_threshold:.2f}")

Heatwave threshold (95th percentile of summer TMAX): 90.00


### 3. Identify heatwave days

In [3]:
# A heatwave is defined as 3 or more consecutive days with TMAX > threshold
df["HOT_DAY"] = (df["TMAX"] > heatwave_threshold).astype(int)

# Mark heatwave days by finding runs of consecutive hot days
df["IS_HEATWAVE_DAY"] = 0

hot = df["HOT_DAY"].values
n = len(df)

start = 0
while start < n:
    if hot[start] == 1:
        end = start
        while end < n and hot[end] == 1:
            end += 1

        run_length = end - start
        if run_length >= 3:
            df.loc[start:end - 1, "IS_HEATWAVE_DAY"] = 1

        start = end
    else:
        start += 1

### 4. Identify heatwave onset days

In [4]:
# Onset day = first day of each heatwave event
df["HEATWAVE_ONSET"] = 0

for i in range(n):
    if df.loc[i, "IS_HEATWAVE_DAY"] == 1:
        if i == 0 or df.loc[i - 1, "IS_HEATWAVE_DAY"] == 0:
            df.loc[i, "HEATWAVE_ONSET"] = 1

### 5. Construct forecasting target

In [5]:
# Target = whether a heatwave will begin within the next 3 days
# y_t = 1 if onset happens in t+1, t+2, or t+3
df["TARGET_ONSET_NEXT_3D"] = 0

for i in range(n):
    future_window = df.loc[i + 1:i + 3, "HEATWAVE_ONSET"]
    if future_window.sum() > 0:
        df.loc[i, "TARGET_ONSET_NEXT_3D"] = 1

# Remove the last 3 rows because they do not have full future information
df = df.iloc[:-3].copy().reset_index(drop=True)

### 6. Feature engineering

In [6]:
# Lag features
df["TMAX_LAG1"] = df["TMAX"].shift(1)
df["TMAX_LAG2"] = df["TMAX"].shift(2)
df["TMAX_LAG3"] = df["TMAX"].shift(3)

df["TMIN_LAG1"] = df["TMIN"].shift(1)
df["TMIN_LAG2"] = df["TMIN"].shift(2)
df["TMIN_LAG3"] = df["TMIN"].shift(3)

df["PRCP_LAG1"] = df["PRCP"].shift(1)
df["PRCP_LAG2"] = df["PRCP"].shift(2)
df["PRCP_LAG3"] = df["PRCP"].shift(3)

# Rolling features using only current/past information
df["TMAX_ROLL3_MEAN"] = df["TMAX"].rolling(window=3).mean()
df["TMAX_ROLL5_MEAN"] = df["TMAX"].rolling(window=5).mean()
df["TMAX_ROLL7_MEAN"] = df["TMAX"].rolling(window=7).mean()

df["TMIN_ROLL3_MEAN"] = df["TMIN"].rolling(window=3).mean()
df["TMIN_ROLL5_MEAN"] = df["TMIN"].rolling(window=5).mean()

df["PRCP_ROLL3_SUM"] = df["PRCP"].rolling(window=3).sum()
df["PRCP_ROLL5_SUM"] = df["PRCP"].rolling(window=5).sum()

# Temperature change features
df["TMAX_DIFF_1"] = df["TMAX"] - df["TMAX"].shift(1)
df["TMIN_DIFF_1"] = df["TMIN"] - df["TMIN"].shift(1)

# Diurnal temperature range
df["TEMP_RANGE"] = df["TMAX"] - df["TMIN"]

# Recent hot-day count based on threshold
df["HOT_DAY_LAST3"] = df["HOT_DAY"].rolling(window=3).sum()
df["HOT_DAY_LAST5"] = df["HOT_DAY"].rolling(window=5).sum()

# Seasonal encoding
df["MONTH_SIN"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * df["MONTH"] / 12)
df["DOY_SIN"] = np.sin(2 * np.pi * df["DAYOFYEAR"] / 365.25)
df["DOY_COS"] = np.cos(2 * np.pi * df["DAYOFYEAR"] / 365.25)

# Drop rows with missing values created by lag/rolling operations
df_model = df.dropna().copy().reset_index(drop=True)

### 7. Prepare train/test split

In [7]:
# Time-based split: train on earlier years, test on later years
train_df = df_model[df_model["YEAR"] <= 2018].copy()
test_df = df_model[df_model["YEAR"] >= 2019].copy()

if len(train_df) == 0 or len(test_df) == 0:
    raise ValueError("Train or test set is empty. Please check your date range and split.")

# Weather indicator column list
wt_cols = [col for col in ["WT01", "WT03", "WT04", "WT05", "WT06", "WT11"] if col in df_model.columns]

feature_cols = [
    "TMAX", "TMIN", "PRCP", "SNOW", "SNWD", "TOBS",
    "TMAX_LAG1", "TMAX_LAG2", "TMAX_LAG3",
    "TMIN_LAG1", "TMIN_LAG2", "TMIN_LAG3",
    "PRCP_LAG1", "PRCP_LAG2", "PRCP_LAG3",
    "TMAX_ROLL3_MEAN", "TMAX_ROLL5_MEAN", "TMAX_ROLL7_MEAN",
    "TMIN_ROLL3_MEAN", "TMIN_ROLL5_MEAN",
    "PRCP_ROLL3_SUM", "PRCP_ROLL5_SUM",
    "TMAX_DIFF_1", "TMIN_DIFF_1",
    "TEMP_RANGE",
    "HOT_DAY_LAST3", "HOT_DAY_LAST5",
    "MONTH_SIN", "MONTH_COS", "DOY_SIN", "DOY_COS"
] + wt_cols

feature_cols = [col for col in feature_cols if col in df_model.columns]

X_train = train_df[feature_cols]
y_train = train_df["TARGET_ONSET_NEXT_3D"]

X_test = test_df[feature_cols]
y_test = test_df["TARGET_ONSET_NEXT_3D"]

print("\nTrain size:", len(train_df))
print("Test size:", len(test_df))
print("Positive rate in train:", y_train.mean())
print("Positive rate in test:", y_test.mean())


Train size: 6670
Test size: 2152
Positive rate in train: 0.0026986506746626685
Positive rate in test: 0.006970260223048327


### (My carbon tracking)

In [ ]:
from codecarbon import EmissionsTracker
codecarbon_available = True

### 8. Logistic Regression model

In [ ]:
log_reg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])


tracker_lr = EmissionsTracker(project_name="logistic_regression_heatwave", log_level="error")
tracker_lr.start()

log_reg_pipeline.fit(X_train, y_train)

emissions_lr = tracker_lr.stop()
print(f"\nEstimated emissions for Logistic Regression: {emissions_lr} kg CO2eq")

lr_pred = log_reg_pipeline.predict(X_test)
lr_prob = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("\n" + "=" * 50)
print("Logistic Regression Results")
print("=" * 50)
print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_pred))
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, digits=4))

try:
    lr_auc = roc_auc_score(y_test, lr_prob)
    print(f"ROC-AUC: {lr_auc:.4f}")
except ValueError:
    print("ROC-AUC could not be computed.")

[codecarbon WARNING @ 14:21:26] Multiple instances of codecarbon are allowed to run at the same time.



Estimated emissions for Logistic Regression: 4.711186506804503e-06 kg CO2eq

Logistic Regression Results
Confusion Matrix:
[[2017  120]
 [  10    5]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9951    0.9438    0.9688      2137
           1     0.0400    0.3333    0.0714        15

    accuracy                         0.9396      2152
   macro avg     0.5175    0.6386    0.5201      2152
weighted avg     0.9884    0.9396    0.9625      2152

ROC-AUC: 0.9180


### 9. Random Forest model

In [13]:
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

if codecarbon_available:
    tracker_rf = EmissionsTracker(project_name="random_forest_heatwave", log_level="error")
    tracker_rf.start()

rf_pipeline.fit(X_train, y_train)

if codecarbon_available:
    emissions_rf = tracker_rf.stop()
    print(f"\nEstimated emissions for Random Forest: {emissions_rf} kg CO2eq")

rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("\n" + "=" * 50)
print("Random Forest Results")
print("=" * 50)
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, digits=4))

try:
    rf_auc = roc_auc_score(y_test, rf_prob)
    print(f"ROC-AUC: {rf_auc:.4f}")
except ValueError:
    print("ROC-AUC could not be computed.")


Estimated emissions for Random Forest: 5.414831555180408e-06 kg CO2eq

Random Forest Results
Confusion Matrix:
[[2081   56]
 [  11    4]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9947    0.9738    0.9842      2137
           1     0.0667    0.2667    0.1067        15

    accuracy                         0.9689      2152
   macro avg     0.5307    0.6202    0.5454      2152
weighted avg     0.9883    0.9689    0.9780      2152

ROC-AUC: 0.7773


### 10. Save outputs

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Save model-ready dataset
df_model.to_csv(output_dir / "processed_heatwave_dataset.csv", index=False)

# Save test predictions
lr_results = test_df[["DATE", "TARGET_ONSET_NEXT_3D"]].copy()
lr_results["LR_PRED"] = lr_pred
lr_results["LR_PROB"] = lr_prob

rf_results = test_df[["DATE", "TARGET_ONSET_NEXT_3D"]].copy()
rf_results["RF_PRED"] = rf_pred
rf_results["RF_PROB"] = rf_prob

lr_results.to_csv(output_dir / "logistic_regression_predictions.csv", index=False)
rf_results.to_csv(output_dir / "random_forest_predictions.csv", index=False)

# Save feature importance for random forest
rf_model = rf_pipeline.named_steps["model"]
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.to_csv(output_dir / "random_forest_feature_importance.csv", index=False)

# Save threshold info
threshold_df = pd.DataFrame({
    "metric": ["heatwave_threshold"],
    "value": [heatwave_threshold]
})
threshold_df.to_csv(output_dir / "heatwave_threshold.csv", index=False)

print("\nFiles saved to:", output_dir.resolve())


Files saved to: C:\Users\Maxwell\Desktop\Courses\5550\5550_project\outputs


### 11. Simple baseline comparison

In [15]:
# Baseline: always predict 0 (no heatwave onset in next 3 days)
baseline_pred = np.zeros(len(y_test), dtype=int)

print("\n" + "=" * 50)
print("Baseline Results (Always Predict 0)")
print("=" * 50)
print("Confusion Matrix:")
print(confusion_matrix(y_test, baseline_pred))
print("\nClassification Report:")
print(classification_report(y_test, baseline_pred, digits=4, zero_division=0))


# 13. End-of-pipeline summary

print("\n" + "=" * 50)
print("Pipeline completed successfully.")
print("=" * 50)
print(f"Original cleaned rows: {len(df)}")
print(f"Model-ready rows: {len(df_model)}")
print(f"Number of heatwave onset days: {df_model['HEATWAVE_ONSET'].sum()}")
print(f"Number of positive target days: {df_model['TARGET_ONSET_NEXT_3D'].sum()}")
print(f"Number of features used: {len(feature_cols)}")


Baseline Results (Always Predict 0)
Confusion Matrix:
[[2137    0]
 [  15    0]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9930    1.0000    0.9965      2137
           1     0.0000    0.0000    0.0000        15

    accuracy                         0.9930      2152
   macro avg     0.4965    0.5000    0.4983      2152
weighted avg     0.9861    0.9930    0.9896      2152


Pipeline completed successfully.
Original cleaned rows: 8834
Model-ready rows: 8822
Number of heatwave onset days: 11
Number of positive target days: 33
Number of features used: 37
